# DP1 Visit Screening Driver

This notebook is the **real-data visit-screening driver** for Rubin DP1 visit images.

It keeps the Butler / TAP loading workflow in the notebook, while using `fittingTools.py` as the analysis layer for:
- observed-star preprocessing
- stamp-level PSF model comparison
- star / detector / visit summary tables
- visit badness ranking and screening plots

The default path is now **full mode** so the notebook runs the visit-screening workflow directly. The notebook still keeps optional debug plotting off by default and focuses on flagged bad visits.


## Operational workflow

Recommended order of use:
1. run the **synthetic benchmark notebook** first when changing fitting logic or metric definitions
2. run this DP1 notebook in **full mode** on the intended visit subset
3. inspect the visit ranking table and the flagged bad-visit diagnostics
4. tune the visit / detector selection knobs only if the run is too large or too small

This notebook is therefore the **operational real-data driver**, not the place to debug basic metric behavior from scratch.


In [90]:
import random
import time
from collections import defaultdict
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

import lsst.geom
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service

cwd = Path.cwd().resolve()
candidate_roots = [
    cwd,
    cwd.parent if cwd.name == "notebooks" else None,
    cwd / "Rubin-PSF-Analysis",
]
repo_root = None
for candidate in candidate_roots:
    if candidate is not None and (candidate / "src" / "fittingTools.py").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Could not locate the Rubin-PSF-Analysis repository root.")

print("Repository root:", repo_root)


Repository root: /Users/jinshuozhang/Library/CloudStorage/Dropbox/Rubin-Work/Rubin-PSF-Analysis


In [91]:
rsp_tap = get_tap_service("tap")
assert rsp_tap is not None

butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

print("Connected to RSP TAP and DP1 Butler.")


Connected to RSP TAP and DP1 Butler.


## 1. Configuration

This notebook is configured to run in **full** mode by default. Keep the detector and visit-selection knobs explicit so the run size stays manageable.

`START_NIGHT_INDEX` and `END_NIGHT_INDEX` slice the chronologically sorted list of available `day_obs` values by **index position**, not by literal `day_obs` number. They are 1-based indices into the available-night list.


In [92]:
ANALYSIS_MODE = "full"

START_NIGHT_INDEX = 1
END_NIGHT_INDEX = 3
VISITS_PER_NIGHT = 5

DETECTOR_MODE = "subset"  # "single", "subset", or "all"
DETECTOR_ID = 0
DETECTOR_IDS = [0, 1, 2, 3]
MAX_DETECTORS_PER_VISIT = None

STARS_PER_DETECTOR = 12

BORDER_WIDTH = 2
SHAPELET_BETA = 2.0
SHAPELET_NMAX = 6
FIT_CENTER = False
FIT_BACKGROUND = False
CORE_RADIUS = 2.0
WING_RADIUS = 3.0

RUN_TABLE_ONLY = True
RUN_BAD_VISIT_PLOTS = False
PLOT_TOP_BAD_VISITS = 3
DEBUG_PLOT_ALL_STARS = False
DEBUG_PAGE_SIZE = 2

SAVE_OUTPUT_TABLES = True
SAVE_MASTER_TABLE = True
SAVE_SHAPELET_COEFF_TABLE = False
OUTPUT_DIR = repo_root / "outputs" / "visit_screening"
OUTPUT_PREFIX = f"dp1_{ANALYSIS_MODE}"

KNOWN_BAD_VISITS = []
KNOWN_MODERATE_VISITS = []
KNOWN_GOOD_VISITS = []
RANDOM_SEED = 42

config_items = {
    "ANALYSIS_MODE": ANALYSIS_MODE,
    "START_NIGHT_INDEX": START_NIGHT_INDEX,
    "END_NIGHT_INDEX": END_NIGHT_INDEX,
    "VISITS_PER_NIGHT": VISITS_PER_NIGHT,
    "DETECTOR_MODE": DETECTOR_MODE,
    "DETECTOR_ID": DETECTOR_ID,
    "DETECTOR_IDS": DETECTOR_IDS,
    "MAX_DETECTORS_PER_VISIT": MAX_DETECTORS_PER_VISIT,
    "STARS_PER_DETECTOR": STARS_PER_DETECTOR,
    "RUN_TABLE_ONLY": RUN_TABLE_ONLY,
    "RUN_BAD_VISIT_PLOTS": RUN_BAD_VISIT_PLOTS,
    "PLOT_TOP_BAD_VISITS": PLOT_TOP_BAD_VISITS,
    "SAVE_OUTPUT_TABLES": SAVE_OUTPUT_TABLES,
    "SAVE_MASTER_TABLE": SAVE_MASTER_TABLE,
    "SAVE_SHAPELET_COEFF_TABLE": SAVE_SHAPELET_COEFF_TABLE,
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "KNOWN_BAD_VISITS": KNOWN_BAD_VISITS,
    "KNOWN_MODERATE_VISITS": KNOWN_MODERATE_VISITS,
    "KNOWN_GOOD_VISITS": KNOWN_GOOD_VISITS,
}

display(pd.DataFrame({"config": list(config_items.keys()), "value": list(config_items.values())}))



,config,value
0,ANALYSIS_MODE,full
1,START_NIGHT_INDEX,1
2,END_NIGHT_INDEX,3
3,VISITS_PER_NIGHT,5
4,DETECTOR_MODE,subset
5,DETECTOR_ID,0
6,DETECTOR_IDS,"[0, 1, 2, 3]"
7,MAX_DETECTORS_PER_VISIT,None
8,STARS_PER_DETECTOR,12
9,PLOT_TOP_BAD_VISITS,3


## 2. Query candidate visits

This step keeps the Butler workflow in the notebook. We first query available `visit_image` references, then select a configurable subset of visits and detectors to screen.


In [ ]:
rng = random.Random(RANDOM_SEED)

all_visit_refs = list(
    butler.query_datasets(
        "visit_image",
        order_by=["day_obs", "visit", "detector"],
        with_dimension_records=True,
    )
)

visit_ref_tree = defaultdict(lambda: defaultdict(dict))
for ref in all_visit_refs:
    day_obs = int(ref.dataId["day_obs"])
    visit_id = int(ref.dataId["visit"])
    detector_id = int(ref.dataId["detector"])
    visit_ref_tree[day_obs][visit_id][detector_id] = ref

all_nights_sorted = sorted(visit_ref_tree.keys())
selected_nights = all_nights_sorted[max(START_NIGHT_INDEX - 1, 0):END_NIGHT_INDEX]

selected_visit_records = []
selected_ref_map = {}
configured_known_visits = sorted(set(KNOWN_BAD_VISITS) | set(KNOWN_MODERATE_VISITS) | set(KNOWN_GOOD_VISITS))
known_visit_lookup = {}
for day_obs, visit_map in visit_ref_tree.items():
    for visit_id in visit_map.keys():
        known_visit_lookup[int(visit_id)] = int(day_obs)

def add_selected_visit_ref(day_obs, visit_id, detector_id, ref):
    pair = (int(visit_id), int(detector_id))
    if pair in selected_ref_map:
        return
    selected_ref_map[pair] = ref
    selected_visit_records.append(
        {
            "day_obs": int(day_obs),
            "visit": int(visit_id),
            "detector": int(detector_id),
            "band": str(ref.dataId["band"]),
            "selection_source": "night_sample",
        }
    )

for day_obs in selected_nights:
    visit_ids = sorted(visit_ref_tree[day_obs].keys())
    if not visit_ids:
        continue

    n_sample = min(VISITS_PER_NIGHT, len(visit_ids))
    sampled_visit_ids = visit_ids if n_sample == len(visit_ids) else sorted(rng.sample(visit_ids, n_sample))

    for visit_id in sampled_visit_ids:
        detector_map = visit_ref_tree[day_obs][visit_id]
        available_detectors = sorted(detector_map.keys())

        if DETECTOR_MODE == "single":
            requested_detectors = [DETECTOR_ID] if DETECTOR_ID in detector_map else []
        elif DETECTOR_MODE == "subset":
            requested_detectors = [det for det in DETECTOR_IDS if det in detector_map]
        elif DETECTOR_MODE == "all":
            requested_detectors = available_detectors
        else:
            raise ValueError(f"Unknown DETECTOR_MODE: {DETECTOR_MODE}")

        if MAX_DETECTORS_PER_VISIT is not None:
            requested_detectors = requested_detectors[:MAX_DETECTORS_PER_VISIT]

        for detector_id in requested_detectors:
            ref = detector_map[detector_id]
            add_selected_visit_ref(day_obs, visit_id, detector_id, ref)

included_known_visits = []
missing_known_visits = []
for visit_id in configured_known_visits:
    day_obs = known_visit_lookup.get(int(visit_id))
    if day_obs is None:
        missing_known_visits.append(int(visit_id))
        continue

    detector_map = visit_ref_tree[day_obs][int(visit_id)]
    available_detectors = sorted(detector_map.keys())
    if DETECTOR_MODE == "single":
        requested_detectors = [DETECTOR_ID] if DETECTOR_ID in detector_map else []
    elif DETECTOR_MODE == "subset":
        requested_detectors = [det for det in DETECTOR_IDS if det in detector_map]
    elif DETECTOR_MODE == "all":
        requested_detectors = available_detectors
    else:
        raise ValueError(f"Unknown DETECTOR_MODE: {DETECTOR_MODE}")

    if MAX_DETECTORS_PER_VISIT is not None:
        requested_detectors = requested_detectors[:MAX_DETECTORS_PER_VISIT]

    if not requested_detectors:
        missing_known_visits.append(int(visit_id))
        continue

    included_known_visits.append(int(visit_id))
    for detector_id in requested_detectors:
        ref = detector_map[detector_id]
        pair = (int(visit_id), int(detector_id))
        if pair not in selected_ref_map:
            selected_ref_map[pair] = ref
            selected_visit_records.append(
                {
                    "day_obs": int(day_obs),
                    "visit": int(visit_id),
                    "detector": int(detector_id),
                    "band": str(ref.dataId["band"]),
                    "selection_source": "known_visit",
                }
            )

selected_visits_df = pd.DataFrame(selected_visit_records).sort_values(["day_obs", "visit", "detector"]).reset_index(drop=True)
print(f"Total nights available: {len(all_nights_sorted)}")
print(f"Selected night-index range: {START_NIGHT_INDEX} to {END_NIGHT_INDEX}")
print(f"Selected day_obs values: {selected_nights}")
if configured_known_visits:
    print(f"Configured known visits: {configured_known_visits}")
    print(f"Included known visits in this run: {sorted(set(included_known_visits))}")
    if missing_known_visits:
        print(f"Known visits not available under current detector selection: {sorted(set(missing_known_visits))}")
print(f"Selected visit-detector refs: {len(selected_visit_records)}")
display(selected_visits_df.head(20))


## 3. Query and sample PSF-used stars

We keep the TAP query here, but move all stamp-level model analysis out to `fittingTools.py`.


In [ ]:
if selected_visit_records:
    selected_visit_ids = sorted({record["visit"] for record in selected_visit_records})
    selected_detector_ids = sorted({record["detector"] for record in selected_visit_records})

    visit_id_list = ", ".join(str(v) for v in selected_visit_ids)
    detector_id_list = ", ".join(str(d) for d in selected_detector_ids)

    query = (
        "SELECT sourceId, x, y, visit, detector "
        "FROM dp1.Source "
        f"WHERE visit IN ({visit_id_list}) "
        f"AND detector IN ({detector_id_list}) "
        "AND calib_psf_used = 1"
    )

    job = rsp_tap.submit_job(query)
    job.run()
    job.wait(phases=["COMPLETED", "ERROR"])
    print("TAP query phase:", job.phase)
    if job.phase == "ERROR":
        job.raise_if_error()

    all_stars_df = job.fetch_result().to_table().to_pandas()
    selected_pair_df = pd.DataFrame(
        [{"visit": visit, "detector": detector} for visit, detector in selected_ref_map.keys()]
    ).drop_duplicates()
    all_stars_df = all_stars_df.merge(selected_pair_df, on=["visit", "detector"], how="inner")
else:
    all_stars_df = pd.DataFrame(columns=["sourceId", "x", "y", "visit", "detector"])

selected_star_map = {}
selected_star_rows = []
for (visit_id, detector_id), group in all_stars_df.groupby(["visit", "detector"], sort=True):
    sample_n = min(STARS_PER_DETECTOR, len(group))
    sampled_group = group.sort_values("sourceId") if sample_n == len(group) else group.sample(n=sample_n, random_state=RANDOM_SEED)
    sampled_group = sampled_group.sort_values("sourceId").reset_index(drop=True)
    selected_star_map[(int(visit_id), int(detector_id))] = sampled_group
    selected_star_rows.append(
        {
            "visit": int(visit_id),
            "detector": int(detector_id),
            "n_psf_candidates": int(len(group)),
            "n_sampled_stars": int(len(sampled_group)),
        }
    )

star_selection_df = pd.DataFrame(selected_star_rows)
print(f"Total sampled stars: {sum(len(df) for df in selected_star_map.values())}")
display(star_selection_df.head(20))


## 4. Analysis backend

The notebook keeps only the data-loading and orchestration logic. All preprocessing, stamp-level fitting, and metric aggregation are delegated to `fittingTools.py`. This includes the star-level `high_ng_star_flag` and the detector / visit-level highly non-Gaussian fractions used for later statistical summaries.


In [ ]:
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import importlib
import simulationTools
importlib.reload(simulationTools)

import fittingTools
importlib.reload(fittingTools)

print("Loaded simulationTools from:", simulationTools.__file__)
print("Loaded fittingTools from:", fittingTools.__file__)


## 5. Load cutouts and analyze observed stars

For each selected visit and detector, load observed star cutouts from `visit_image`, then pass each stamp to `fittingTools.analyze_observed_star(...)`.


In [ ]:
analysis_results = []
analysis_start = time.time()

for record in selected_visit_records:
    visit_id = record["visit"]
    detector_id = record["detector"]
    band = record["band"]
    day_obs = record["day_obs"]
    pair = (visit_id, detector_id)

    star_table = selected_star_map.get(pair)
    if star_table is None or star_table.empty:
        print(f"Visit {visit_id}, detector {detector_id}: no sampled stars")
        continue

    visit_image = butler.get(selected_ref_map[pair])
    psf = visit_image.getPsf()

    print(f"Analyzing visit {visit_id}, detector {detector_id}, band {band}: {len(star_table)} stars")

    for _, star in star_table.iterrows():
        x = float(star["x"])
        y = float(star["y"])
        source_id = int(star["sourceId"])
        position = lsst.geom.Point2D(x, y)

        try:
            psf_stamp = psf.computeImage(position)
            stamp_ny, stamp_nx = psf_stamp.array.shape
            extent = lsst.geom.Extent2I(int(stamp_nx), int(stamp_ny))
            cutout = visit_image.getCutout(position, extent)
            stamp = cutout.getMaskedImage().image.array

            result = fittingTools.analyze_observed_star(
                stamp,
                x=x,
                y=y,
                visit=visit_id,
                detector=detector_id,
                band=band,
                day_obs=day_obs,
                center=None,
                border_width=BORDER_WIDTH,
                shapelet_beta=SHAPELET_BETA,
                shapelet_nmax=SHAPELET_NMAX,
                fit_center=FIT_CENTER,
                fit_background=FIT_BACKGROUND,
                core_radius=CORE_RADIUS,
                wing_radius=WING_RADIUS,
            )
            result["sourceId"] = source_id
            result["rubin_stamp_nx"] = int(stamp_nx)
            result["rubin_stamp_ny"] = int(stamp_ny)
        except Exception as exc:
            result = {
                "visit": visit_id,
                "detector": detector_id,
                "band": band,
                "day_obs": day_obs,
                "x": x,
                "y": y,
                "sourceId": source_id,
                "preprocess_valid": False,
                "analysis_valid": False,
                "best_model": "none",
                "ng_score": np.nan,
                "error_message": str(exc),
            }

        analysis_results.append(result)

analysis_time = time.time() - analysis_start
print(f"Completed analysis for {len(analysis_results)} stars in {analysis_time:.2f} s")

analysis_overview_df = pd.DataFrame(
    [
        {
            "visit": result.get("visit"),
            "detector": result.get("detector"),
            "band": result.get("band"),
            "analysis_valid": result.get("analysis_valid", False),
            "best_model": result.get("best_model"),
            "ng_score": result.get("ng_score"),
        }
        for result in analysis_results
    ]
)
display(analysis_overview_df.head(20))


## 6. Build and persist summary tables

The screening workflow produces three linked data products:
- `star_metrics_df`
- `detector_metrics_df`
- `visit_metrics_df`

These are the main outputs for later screening, ranking, and validation.


In [ ]:
star_summary_rows = []
for result in analysis_results:
    row = fittingTools.build_star_summary_row(result)
    row["sourceId"] = result.get("sourceId")
    row["error_message"] = result.get("error_message")
    star_summary_rows.append(row)

star_metrics_df = pd.DataFrame(star_summary_rows)
star_metrics_df = fittingTools.annotate_star_quality(star_metrics_df)
detector_metrics_df = fittingTools.summarize_detector_metrics(star_metrics_df)
visit_metrics_df = fittingTools.summarize_visit_metrics(detector_metrics_df)
ranked_visits_df = visit_metrics_df.sort_values("visit_badness", ascending=False).reset_index(drop=True)

if SAVE_SHAPELET_COEFF_TABLE:
    shapelet_coefficients_long_df = fittingTools.build_shapelet_coefficients_long_table(analysis_results)
else:
    shapelet_coefficients_long_df = pd.DataFrame(
        columns=["sourceId", "visit", "detector", "band", "day_obs", "mode_index", "n1", "n2", "coeff"]
    )

if SAVE_OUTPUT_TABLES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    if SAVE_MASTER_TABLE:
        star_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_star_metrics.csv"
        star_metrics_df.to_csv(star_path, index=False)
        print("Saved master star table:")
        print("  ", star_path)

    detector_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_detector_metrics.csv"
    visit_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_visit_metrics.csv"
    detector_metrics_df.to_csv(detector_path, index=False)
    ranked_visits_df.to_csv(visit_path, index=False)
    print("Saved detector / visit tables:")
    print("  ", detector_path)
    print("  ", visit_path)

    if SAVE_SHAPELET_COEFF_TABLE:
        shapelet_coeff_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_shapelet_coefficients_long.csv"
        shapelet_coefficients_long_df.to_csv(shapelet_coeff_path, index=False)
        print("Saved shapelet coefficient sidecar table:")
        print("  ", shapelet_coeff_path)

display(star_metrics_df.head(10))
display(detector_metrics_df.head(10))
display(
    ranked_visits_df[
        [
            "visit",
            "day_obs",
            "band",
            "visit_badness",
            "size_excursion",
            "shape_excursion",
            "non_gaussian_excursion",
            "edg_deviation_excursion",
            "spatial_nonuniformity",
            "median_edg_deviation_score",
            "frac_failed_detectors",
            "frac_science_bad_detectors",
            "frac_high_ng_detectors",
            "frac_high_ng_stars",
            "frac_high_edg_deviation_detectors",
            "frac_high_edg_deviation_stars",
            "bad_visit_flag",
            "bad_visit_rank",
        ]
    ].head(20)
)

if SAVE_SHAPELET_COEFF_TABLE and not shapelet_coefficients_long_df.empty:
    display(shapelet_coefficients_long_df.head(10))



### Master table schema check

This is a compact schema view of the star-level master table. It emphasizes metadata, observed metrics, per-model chi2 values, flattened model parameters, flags, and the compact shapelet summaries used for large-sample analysis.


In [ ]:
star_schema_groups = {
    "metadata": ["sourceId", "visit", "detector", "band", "day_obs", "x", "y"],
    "observed_metrics": ["peak_value", "total_flux", "fwhm_proxy", "ee80_radius", "e1", "e2", "ellipticity"],
    "model_chi2": ["gaussian_chi2", "dg_chi2", "edg_chi2", "gh_chi2", "moffat_chi2", "shapelet_chi2"],
    "flattened_model_params": [
        "gaussian_A", "gaussian_sigma",
        "dg_a1", "dg_sigma1", "dg_a2", "dg_sigma2",
        "edg_A1", "edg_sigma1_x", "edg_sigma1_y", "edg_A2", "edg_sigma2_x", "edg_sigma2_y", "edg_theta",
        "gh_A", "gh_sigma_x", "gh_sigma_y", "gh_h3x", "gh_h4x", "gh_h3y", "gh_h4y",
        "moffat_A", "moffat_alpha", "moffat_beta",
        "shapelet_beta", "shapelet_nmax",
        "shapelet_order0_norm", "shapelet_order1_norm", "shapelet_order2_norm", "shapelet_order3_norm", "shapelet_order4_norm", "shapelet_coeff_l2_total",
    ],
    "flags": [
        "best_model", "ng_score", "edg_deviation_score",
        "failed_star_flag", "science_bad_star_flag", "high_ng_star_flag", "high_edg_deviation_star_flag",
    ],
}

schema_rows = []
for group_name, columns in star_schema_groups.items():
    for column in columns:
        schema_rows.append(
            {
                "group": group_name,
                "column": column in star_metrics_df.columns and column or column,
                "present": column in star_metrics_df.columns,
                "dtype": str(star_metrics_df[column].dtype) if column in star_metrics_df.columns else "missing",
            }
        )

star_schema_df = pd.DataFrame(schema_rows)
print(f"Star master table columns: {star_metrics_df.shape[1]}")
display(star_schema_df)


## 7. Rank visits and plot only flagged bad cases

The default behavior is to rank visits by `visit_badness`, summarize the fraction of flagged bad visits, and only show compact diagnostics for flagged / worst visits. Full per-star galleries are optional debug output and stay off by default.

This section also summarizes the operational non-Gaussian statistic directly, so the notebook can answer questions like:
- what fraction of sampled stars were highly non-Gaussian?
- what fraction of detectors within a visit were highly non-Gaussian?


In [ ]:
if ranked_visits_df.empty:
    print("No visit summaries available.")
else:
    n_flagged = int(ranked_visits_df["bad_visit_flag"].sum())
    print(f"Flagged bad visits: {n_flagged} / {len(ranked_visits_df)}")

    total_stars = len(star_metrics_df)
    n_high_ng_stars = int(star_metrics_df.get("high_ng_star_flag", pd.Series(False)).fillna(False).sum())
    frac_high_ng_stars = n_high_ng_stars / total_stars if total_stars > 0 else np.nan
    print(f"Highly non-Gaussian stars: {n_high_ng_stars} / {total_stars} ({frac_high_ng_stars:.1%})")

    n_high_edg_stars = int(star_metrics_df.get("high_edg_deviation_star_flag", pd.Series(False)).fillna(False).sum())
    frac_high_edg_stars = n_high_edg_stars / total_stars if total_stars > 0 else np.nan
    print(f"Highly EDG-deviant stars: {n_high_edg_stars} / {total_stars} ({frac_high_edg_stars:.1%})")

    print("Top ranked visits:")
    display(
        ranked_visits_df[
            [
                "visit", "day_obs", "band", "visit_badness", "size_excursion", "shape_excursion",
                "non_gaussian_excursion", "edg_deviation_excursion", "spatial_nonuniformity",
                "median_edg_deviation_score", "frac_failed_detectors", "frac_science_bad_detectors",
                "frac_high_ng_detectors", "frac_high_edg_deviation_detectors", "bad_visit_flag", "bad_visit_rank",
            ]
        ].head(20)
    )

    night_summary_df = fittingTools.build_night_summary_table(ranked_visits_df)
    print("Night summary (including EDG-aware aggregates):")
    display(night_summary_df)

    if RUN_TABLE_ONLY or not RUN_BAD_VISIT_PLOTS:
        print("Table-first mode: skipping bad-visit plots.")
    else:
        fittingTools.plot_visit_badness(ranked_visits_df, top_n=max(PLOT_TOP_BAD_VISITS, min(20, len(ranked_visits_df))))

        gallery_visit_df = ranked_visits_df[ranked_visits_df["bad_visit_flag"]].copy()
        if gallery_visit_df.empty:
            gallery_visit_df = ranked_visits_df.head(PLOT_TOP_BAD_VISITS)

        if PLOT_TOP_BAD_VISITS > 0 and not gallery_visit_df.empty:
            fittingTools.plot_bad_visit_gallery(
                analysis_results,
                gallery_visit_df,
                top_n_visits=PLOT_TOP_BAD_VISITS,
                n_examples=max(1, STARS_PER_DETECTOR),
            )

        if DEBUG_PLOT_ALL_STARS:
            grouped_results = defaultdict(list)
            for result in analysis_results:
                if result.get("analysis_valid", False):
                    grouped_results[(result.get("visit"), result.get("detector"), result.get("band"))].append(result)

            for (visit_id, detector_id, band), group_results in grouped_results.items():
                fittingTools.plot_model_comparison_pages(
                    group_results,
                    page_size=DEBUG_PAGE_SIZE,
                    visit_id=visit_id,
                    detector_id=detector_id,
                    band=band,
                )



## 8. Validation checks with known visits

Operationally, this is the first real-data validation harness for the movie visits.

When `KNOWN_BAD_VISITS`, `KNOWN_MODERATE_VISITS`, or `KNOWN_GOOD_VISITS` are configured, the notebook now **actively tries to include those visits in the analyzed sample** (subject to the current detector-selection rules).

The intended test is:
- **known bad visits should rank high** and should usually be flagged
- **known good visits should rank low** and should usually remain unflagged
- once this ordering looks sensible, the same pipeline can be scaled to larger subsets of nights


In [ ]:
validation_visit_ids = sorted(set(KNOWN_BAD_VISITS) | set(KNOWN_MODERATE_VISITS) | set(KNOWN_GOOD_VISITS))

if not validation_visit_ids:
    print("No KNOWN_BAD_VISITS / KNOWN_MODERATE_VISITS / KNOWN_GOOD_VISITS configured.")
elif ranked_visits_df.empty:
    print("No ranked visits available for validation.")
else:
    validation_df = fittingTools.build_known_visit_validation_table(
        ranked_visits_df,
        known_bad_visits=KNOWN_BAD_VISITS,
        known_good_visits=KNOWN_GOOD_VISITS,
        known_moderate_visits=KNOWN_MODERATE_VISITS,
    )

    print("Known-visit validation table:")
    display(validation_df)

    matched_validation_df = validation_df[validation_df["visit_badness"].notna()].copy()
    print(f"Matched configured validation visits: {len(matched_validation_df)} / {len(validation_df)}")
    if not matched_validation_df.empty:
        flagged_rate = matched_validation_df["bad_visit_flag"].fillna(False).mean()
        print(f"Flagged fraction among matched validation visits: {flagged_rate:.1%}")

    known_bad_df = validation_df[validation_df["expected_label"] == "known_bad"].sort_values("visit_badness", ascending=False, na_position="last")
    known_moderate_df = validation_df[validation_df["expected_label"] == "known_moderate"].sort_values("visit_badness", ascending=False, na_position="last")
    known_good_df = validation_df[validation_df["expected_label"] == "known_good"].sort_values("visit_badness", ascending=False, na_position="last")

    if not known_bad_df.empty:
        print("Known bad visits:")
        display(known_bad_df)
    if not known_moderate_df.empty:
        print("Known moderate visits:")
        display(known_moderate_df)
    if not known_good_df.empty:
        print("Known good visits:")
        display(known_good_df)

    missing_visits = validation_df.loc[validation_df["visit_badness"].isna(), "visit"].tolist()
    if missing_visits:
        print("Configured validation visits not present in the analyzed sample:", missing_visits)

